# Load data


In [1]:
import pandas as pd
import numpy as np
import pandas as pd

# Show all text in columns
import pandas as pd

# Show all columns (no ellipsis)
pd.set_option('display.max_columns', None)

# # Show full content of each cell (no truncation of text)
# pd.set_option('display.max_colwidth', None)

# # Optional: adjust display width to avoid line wrapping
# pd.set_option('display.width', None)

# Dompet_updated_cleaned

In [182]:
df=pd.read_csv("/Users/claranatalies/Documents/year 4/FYP/COMP4501-FYP/cleaned_dataset_final/Dompet_updated_cleaned.csv")
df.brand.unique()

/var/folders/8z/b64380y12kv332c9qsmc494w0000gq/T/ipykernel_62833/3821312763.py:1: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("/Users/claranatalies/Documents/year 4/FYP/COMP4501-FYP/cleaned_dataset_final/Dompet_updated_cleaned.csv")


array([nan, 'Lock', 'JIMS HONEY', 'LOOK BACK', 'Baellerry', 'Menbense',
       'Sophie Paris', 'Ceka', 'Disney', 'CaseMe', 'LA LOUIS ANDREANO',
       'Walker Leather', 'Wallts Wallet Goods', 'Smile Goddess', 'CARDIN',
       'Fansy', 'Ratu Bilqis', 'smiggle', 'Samsung', 'Crocodile', 'PIOMA',
       'Oppo', 'Sony', 'CHIBAO', 'ANTAM', 'Miniso', 'Ray Clayn',
       'Forever Young'], dtype=object)

In [183]:
import pandas as pd
import re

# -------------------------------------------------------------------
# 1. Clean seller_chat_time_reply to numeric hours
# -------------------------------------------------------------------

def clean_chat_time(val):
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if s == 'hitungan jam':
        return 1.0
    if s == 'hitungan menit':
        return 0.5
    match = re.search(r'([\d,\.]+)\s*(jam|menit|hours?|hrs?)', s)
    if match:
        num_str = match.group(1).replace(',', '.')
        try:
            num = float(num_str)
        except:
            return None
        unit = match.group(2)
        if unit.startswith('jam') or unit.startswith('hour') or unit.startswith('hr'):
            return num
        elif unit.startswith('menit'):
            return num / 60.0
        else:
            return None
    try:
        return float(s)
    except:
        return None

df['seller_chat_time_reply'] = df['seller_chat_time_reply'].apply(clean_chat_time)

# -------------------------------------------------------------------
# 2. Clean seller_joined_date to numeric years (relative to 2026-03-21)
# -------------------------------------------------------------------

REF_DATE = pd.to_datetime('2026-03-21')

def parse_absolute(s):
    s_clean = re.sub(r'\s*GMT.*$', '', s)
    s_clean = re.sub(r'\s*\(.*$', '', s_clean)
    try:
        return pd.to_datetime(s_clean, errors='coerce')
    except:
        return pd.NaT

def relative_to_years(s):
    match = re.match(r'(\d+)\s+(tahun|bulan|hari)\s+lalu', s)
    if not match:
        return None
    num = int(match.group(1))
    unit = match.group(2)
    if unit == 'tahun':
        return float(num)
    elif unit == 'bulan':
        return num / 12.0
    else:  # hari
        return num / 365.25

def convert_seller_joined_date(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    if 'lalu' in val_str:
        years = relative_to_years(val_str)
        if years is not None:
            return years
    dt = parse_absolute(val_str)
    if pd.notna(dt):
        return (REF_DATE - dt).days / 365.25
    return None

df['seller_joined_date'] = df['seller_joined_date'].apply(convert_seller_joined_date)



In [184]:
import pandas as pd
import re

# ---------- 1. Normalization ----------
def normalize_motif(s):
    if pd.isna(s):
        return ''
    s = str(s).strip().lower()
    s = re.sub(r'[_-]', ' ', s)          # replace dashes/underscores with space
    s = ' '.join(s.split())              # remove extra spaces
    return s

# ---------- 2. Keyword → category mapping (English) ----------
keyword_to_category = {
    # Solid (Polos)
    'polos': 'Solid',
    'solid color': 'Solid',
    'solid': 'Solid',
    'plain': 'Solid',
    'duo tone': 'Solid',
    'duotone': 'Solid',
    '2 warna': 'Solid',

    # Stripes (Garis)
    'garis-garis': 'Stripes',
    'garis': 'Stripes',
    'stripe': 'Stripes',
    'stripes': 'Stripes',

    # Checks (Kotak)
    'kotak-kotak': 'Checks',
    'kotak kotak': 'Checks',
    'kotak': 'Checks',
    'plaid': 'Checks',
    'check': 'Checks',
    'houndstooth': 'Checks',

    # Floral (Bunga)
    'bunga-bunga': 'Floral',
    'bunga': 'Floral',
    'flower': 'Floral',
    'floral': 'Floral',
    'hawai': 'Floral',

    # Print/Logo (Grafis)
    'logo': 'Print/Logo',
    'logo jh': 'Print/Logo',
    'tulisan brand': 'Print/Logo',
    'grafik': 'Print/Logo',
    'graphic': 'Print/Logo',
    'print': 'Print/Logo',
    'sablon': 'Print/Logo',
    'bergambar': 'Print/Logo',
    'losantos': 'Print/Logo',
    'throwup': 'Print/Logo',
    'coach': 'Print/Logo',
    'independent': 'Print/Logo',
    'flag': 'Print/Logo',

    # Other (catch‑all)
    'loreng': 'Other',
    'croco': 'Other',
    'crocodile': 'Other',
    'kucing': 'Other',
    'bulu': 'Other',
    'embos': 'Other',
    'quilting': 'Other',
    'quiltet': 'Other',
    'kulit jeruk': 'Other',
    'jeruk': 'Other',
    'motif jeruk': 'Other',
    'urat kayu': 'Other',
    'serat kayu': 'Other',
    'carbon': 'Other',
    'tikar': 'Other',
    'full grain': 'Other',
    'kulit sintetis': 'Other',
    'tekstur': 'Other',
    'batik': 'Other',
    'songket': 'Other',
    'jacquard songket': 'Other',
    'primer jacquard': 'Other',
    'sulaman': 'Other',
    'hiasan': 'Other',
    'polkadot': 'Other',
    'sprinkle': 'Other',
    'mozaik': 'Other',
    'pattern': 'Other',
    'motive': 'Other',
    'doraemon': 'Other',
    'boneka': 'Other',
    'spiderman red': 'Other',
}

# ---------- 3. Get category for a single motif ----------
def get_category(motif):
    # exact match first
    if motif in keyword_to_category:
        return keyword_to_category[motif]
    # fallback: check if motif contains any known keyword
    for kw, cat in keyword_to_category.items():
        if kw in motif:
            return cat
    return 'Other'

# ---------- 4. Process the column (take first motif) ----------
def clean_motif_column(series):
    norm = series.apply(normalize_motif)
    first_motif = norm.apply(lambda x: x.split(',')[0].strip() if x else '')
    return first_motif.apply(lambda x: get_category(x) if x else None)

# ---------- 5. Apply to your DataFrame ----------
# Replace 'motif' with the actual column name
df['Motif'] = clean_motif_column(df['Motif'])

# ---------- 6. Check results ----------
print(df['Motif'].value_counts())

Motif
Solid         2047
Print/Logo     760
Other          340
Floral          89
Checks          85
Stripes         58
Name: count, dtype: int64


In [185]:
import pandas as pd

# Step 1: Convert the column to lowercase for consistent matching
df['Jenis Garansi'] = df['Jenis Garansi'].str.lower()

# Step 2: Define mapping from Indonesian variants to English categories
mapping = {
    # No Warranty
    'tidak garansi': 'No Warranty',
    'tidak ada': 'No Warranty',
    '-': 'No Warranty',
    '1 minggu': 'No Warranty',
    'garansi 100% original anker': 'No Warranty',

    # Supplier Warranty
    'garansi supplier': 'Supplier Warranty',

    # Seller Warranty
    'garansi seller': 'Seller Warranty',
    'seller': 'Seller Warranty',
    'garansi toko': 'Seller Warranty',

    # Manufacturer Warranty
    'garansi produsen': 'Manufacturer Warranty',

    # International Warranty
    'garansi internasional': 'International Warranty'
}

# Step 3: Apply mapping; any unmapped value defaults to 'No Warranty'
df['Jenis Garansi'] = df['Jenis Garansi'].map(mapping).fillna('No Warranty')

# Step 4: Verify the result
print(df['Jenis Garansi'].value_counts())

Jenis Garansi
No Warranty               7005
Supplier Warranty          796
Manufacturer Warranty      355
Seller Warranty             50
International Warranty      32
Name: count, dtype: int64


In [186]:
df.Bahan.unique()

array(['Kulit Sintetis', nan, 'PU', 'Kanvas', 'PU Leather', 'Kulit, TPU',
       'Kulit', 'Poliester', 'Kanvas, Poliester', 'TPU', 'Kulit sintetis',
       'Nilon', 'Kulit PU', 'Kulit, TPU, Polyurethane', 'Sintetis',
       'Premium PU Leather', 'Polyurethane', 'Kulit, Kulit Sintetis',
       'Polyester Cordura', 'Kulit, Sintetis', 'Katun, Kanvas', 'Bimo',
       'Kulit, Kulit Sintetis, PVC, Sintetis, Lainnya', 'Croco',
       'Cordura', 'Codura', 'Lainnya, Nilon', 'Kulit Sintetis Grade A',
       'Chocoly anti air', 'Katun, Poliester', 'Cordura 300', 'Silikon',
       'PVC', 'Nilon, polyester', 'Kulit sintetis grade A',
       'Bimo 1682 (Waterproof)', 'Chocoly', 'Katun', 'Duralite Nylon',
       'BIMO WATERPROOF', 'Kulit Sintetis, Lainnya', 'Sintetis, Chocoly',
       'PU LEATHER', 'LEATHER PU', 'Bronze', 'Kanvas, Kulit Sintetis',
       'Lainnya', 'Kulit Sintetis, PVC', 'Kulit, Kulit Sintetis, Lainnya',
       'Rayon', 'Chocoly (tahan air)', 'Bimo Waterproof', 'PU leather',
       '

In [187]:
import pandas as pd
import re

def categorize_material(val):
    if pd.isna(val):
        return 'Special Texture & Others'   # or keep NaN, but group with Others
    
    s = str(val).strip().lower()
    
    # ---- Category 1: Genuine Leather ----
    # Condition: contains "kulit" but NO synthetic/PU indicators, OR contains genuine-specific keywords
    synthetic_indicators = ['sintetis', 'pu', 'synthetic', 'faux', 'polyurethane', 'polyutherane']
    if ('kulit' in s and not any(ind in s for ind in synthetic_indicators)) or \
       any(kw in s for kw in ['genuine', 'full grain', 'safiano', 'horse oil']):
        return 'Genuine Leather'
    
    # ---- Category 2: Synthetic Leather ----
    synthetic_keywords = ['pu', 'sintetis', 'sintesis', 'synthetic', 'faux', 
                          'polyurethane', 'polyutherane', 'premium pu']
    if any(kw in s for kw in synthetic_keywords):
        return 'Synthetic Leather'
    
    # ---- Category 3: Technical & Waterproof ----
    tech_keywords = ['bimo', 'bino', 'chocoly', 'cordura', 'codura', 'duralite', 
                     'dinier', 'waterproof', 'ripstock']
    if any(kw in s for kw in tech_keywords):
        return 'Technical & Waterproof'
    
    # ---- Category 4: Fabric & Textile ----
    fabric_keywords = ['kanvas', 'canvas', 'poliester', 'polyester', 'nilon', 'nylon', 
                       'katun', 'rayon', 'denim', 'sutra', 'brokat', 'tapis', 'felt', 'tekstil']
    if any(kw in s for kw in fabric_keywords):
        return 'Fabric & Textile'
    
    # ---- Category 5: Hardware & Hard Materials ----
    hardware_keywords = ['pvc', 'tpu', 'silikon', 'alumunium', 'alloy', 'metal', 
                         'plastik', 'mika', 'logam']
    if any(kw in s for kw in hardware_keywords):
        return 'Hardware & Hard Materials'
    
    # ---- Category 6: Special Texture & Others ----
    # Catch-all for remaining keywords and anything else
    other_keywords = ['croco', 'crocodile', 'kulit jeruk', 'bulu', 'urat kayu', 
                      'mozaik', 'bronze', 'balsam', 'lainnya', 'nan']
    if any(kw in s for kw in other_keywords):
        return 'Special Texture & Others'
    
    # Default fallback
    return 'Special Texture & Others'

# Apply to your column
# Replace 'material' with the actual column name
df['Bahan'] = df['Bahan'].apply(categorize_material)

# Check distribution
print(df['Bahan'].value_counts())

Bahan
Special Texture & Others     3447
Synthetic Leather            3063
Fabric & Textile              920
Genuine Leather               538
Technical & Waterproof        232
Hardware & Hard Materials      38
Name: count, dtype: int64


In [188]:
df['Masa Garansi'].unique()

array(['Tidak Garansi', nan, '1 Minggu', '1 Bulan', '24 JAM', '3 Hari',
       '3 Bulan', '14 Hari', '6 Bulan', '2 Bulan', '2 Minggu',
       'GARANSI SHOPEE', '12 Bulan', 'TIDAK GARANSI', '2 MINGGU',
       '24 jam', '1 minggu', '3 hari setelah barang di terima',
       '1 MINGGU', '3 Hari Sejak barang diterima', 'Garansi Shopee',
       'garansi shopee', '1 x 24 jam sejak produk diterima',
       '3 Hari Setelah Barang Diterima', 'Tidak Ada',
       '3 hari setalah barang di kirim', '3 HARI', 'Garansi shopee',
       '5 Tahun', 'TIDAK ADA', '1 HARI SETELAH BARANG DITERIMA', '3 hari',
       'Tidak ada garansi', 'Tidak ada', 'Tidak Ada Garansi',
       'TDK GARANSI', '2 minggu', '7 Hari',
       'tidak tersedia garansi produk', 'TIDAK ADA GARANSI',
       '3 Hari setelah barang di terima', '1 MInggu', '0',
       '3 Hari Setelah Barang DIterima',
       '1 HARI SETELAH BARANG DI TERIMA', 'garansi penjual',
       '1x24 jam S&k berlaku', '24Jam', '3 hari setelah terima barang',
       

In [189]:
import pandas as pd
import re

def categorize_warranty(val):
    # 1. Handle missing values
    if pd.isna(val):
        return 'No Warranty'

    s = str(val).strip().lower()

    # 2. No Warranty keywords
    no_warranty = [
        'tidak garansi', 'tidak ada', '0', 'tdk garansi',
        'tidak tersedia', 'tidak ada garansi'
    ]
    if any(kw in s for kw in no_warranty):
        return 'No Warranty'

    # 3. Platform / Seller warranties → Short Term
    if 'shopee' in s or 'penjual' in s:
        return 'Short Term'

    # 4. Parse numeric duration
    match = re.search(r'(\d+(?:\.\d+)?)\s*(hari|jam|minggu|bulan|tahun)', s)
    if match:
        num = float(match.group(1))
        unit = match.group(2)

        # Convert to days for comparison
        if unit == 'jam':
            days = num / 24.0
        elif unit == 'hari':
            days = num
        elif unit == 'minggu':
            days = num * 7
        elif unit == 'bulan':
            days = num * 30.44
        else:  # tahun
            days = num * 365.25

        # Categorize by duration
        if unit in ['hari', 'jam'] and days <= 3:
            return 'Immediate'
        elif unit == 'minggu':
            return 'Short Term'
        elif unit == 'hari' and days > 3:
            return 'Short Term'
        elif unit == 'bulan' and num <= 6:
            return 'Medium Term'
        elif unit == 'bulan' and num > 6:
            return 'Long Term'
        elif unit == 'tahun':
            return 'Long Term'

    # 5. Fallback
    return 'No Warranty'

# Apply to your DataFrame column
# Replace 'warranty' with the actual column name
df['Masa Garansi'] = df['Masa Garansi'].apply(categorize_warranty)

# Check distribution
print(df['Masa Garansi'].value_counts())

Masa Garansi
No Warranty    7083
Medium Term     701
Short Term      331
Immediate        81
Long Term        42
Name: count, dtype: int64


In [190]:
df['Negara Asal'].unique()

array(['Indonesia', nan, 'China', 'Korea', 'Taiwan', 'Lainnya', 'Amerika',
       'Import', 'Bangladesh', 'Vietnam', 'Jepang', 'Lokal', 'India'],
      dtype=object)

In [191]:
df['Negara Asal'] = df['Negara Asal'].fillna('Tidak Diketahui')

In [192]:
df['Penutup Dompet'].unique()

array(['Lipat', nan, 'Resleting', 'Lainnya', 'Kancing', 'Kunci Pin',
       'Magnetic Clousure', 'magnet', 'Velcro', 'magnetic',
       'Resleting dan kancing', 'Kancing Maget', 'Tutup klip',
       'Penutup magnet', 'Kunci', 'RESLETING + KONCI SODOK', 'Magnet',
       'Gawang', 'Kancing Magnet', 'kancing dan risleting',
       'Kancing dan resleting', 'Resleting dan Kancing', 'MAGNET'],
      dtype=object)

In [193]:
import pandas as pd
import re

def categorize_closure(val):
    if pd.isna(val):
        return 'Other'
    
    s = str(val).strip().lower()
    
    # Define keyword groups
    zipper = ['resleting', 'risleting', 'zip']
    button = ['kancing']
    magnetic = ['magnet', 'magnetic']
    snap_hook = ['kunci', 'pin', 'klip', 'gawang', 'tutup klip']
    velcro = ['velcro']
    other = ['lipat', 'lainnya']   # plus unmatched
    
    # Count how many categories are matched
    matched = []
    if any(kw in s for kw in zipper):
        matched.append('Zipper')
    if any(kw in s for kw in button):
        matched.append('Button')
    if any(kw in s for kw in magnetic):
        matched.append('Magnetic')
    if any(kw in s for kw in snap_hook):
        matched.append('Snap/Hook/Lock')
    if any(kw in s for kw in velcro):
        matched.append('Velcro')
    
    # If multiple categories matched -> Combination
    if len(matched) > 1:
        return 'Combination'
    elif len(matched) == 1:
        return matched[0]
    else:
        # Check explicit 'other' keywords, else default to 'Other'
        if any(kw in s for kw in other):
            return 'Other'
        else:
            return 'Other'   # catch‑all
        
df['Penutup Dompet'] = df['Penutup Dompet'].apply(categorize_closure)

# Check distribution
print(df['Penutup Dompet'].value_counts())

Penutup Dompet
Other             6420
Button             976
Zipper             721
Magnetic            56
Velcro              28
Combination         23
Snap/Hook/Lock      14
Name: count, dtype: int64


In [194]:
df['Penutup Dompet'].unique()

array(['Other', 'Zipper', 'Button', 'Snap/Hook/Lock', 'Magnetic',
       'Velcro', 'Combination'], dtype=object)

In [195]:
df['Jumlah Slot Kartu'] = df['Jumlah Slot Kartu'].fillna(0)

In [196]:
df['Jumlah Slot Kartu'].unique()

array(['6', 0, '8', '4', '> 10', '2', '10'], dtype=object)

In [197]:
df['Slot Koin'] = df['Slot Koin'].fillna('Tidak')
df['Slot Koin'].value_counts()

Slot Koin
Tidak    6751
Ya       1487
Name: count, dtype: int64

In [198]:
df.head()


,id,title,rating,reviews,initial_price,final_price,stock,favorite,seller_name,breadcrumb,seller_rating,seller_products,seller_chats_responded_percentage,seller_chat_time_reply,seller_joined_date,seller_followers,sold,brand,gmv_cal,Motif,Jenis Garansi,Bahan,Masa Garansi,Negara Asal,Penutup Dompet,Jumlah Slot Kartu,Slot Koin,Tekstur Kulit,Tampilan Kulit,Jenis Kulit,Stok,Dikirim Dari,variations_count,voucher_status,image_count,video_count,review,discount,breadcrumb_Mobile & Tablet Accessories,breadcrumb_Automotive Keychains & Key Covers,breadcrumb_Other Fashion Accessories,breadcrumb_Tote Bags,breadcrumb_Sports & Outdoor,breadcrumb_Other Sports & Outdoor,breadcrumb_Women Shoes,breadcrumb_Tas & Koper,breadcrumb_Aksesoris Make Up,breadcrumb_Umbrella,breadcrumb_Aksesoris Bayi & Anak,breadcrumb_Waist Bags & Chest Bags,breadcrumb_Motorcycle Rider Accessories,breadcrumb_Dompet Wanita,breadcrumb_Logam Mulia,breadcrumb_Sets,breadcrumb_Tas Wanita,breadcrumb_Women Bags,breadcrumb_Aksesoris Konsol,breadcrumb_Fashion Bayi & Anak,breadcrumb_Bag Accessories,breadcrumb_Handphone & Aksesoris,breadcrumb_Boy Bags,breadcrumb_Men Shoes,breadcrumb_Dompet,breadcrumb_Alat Kecantikan,breadcrumb_Ear Care,breadcrumb_Muslim Fashion,breadcrumb_Clutches & Wristlets,breadcrumb_Home & Living,breadcrumb_Health,breadcrumb_Aksesoris Fashion Lainnya,breadcrumb_Mobile & Accessories,breadcrumb_First Aid Supplies,breadcrumb_Eyes Makeup,breadcrumb_Other Photography,breadcrumb_Needlework,breadcrumb_Souvenir & Perlengkapan Pesta,breadcrumb_Stationery & Books,breadcrumb_Beauty & Care,breadcrumb_Men Clothes,breadcrumb_Casing & Skin,breadcrumb_Tas Pria,breadcrumb_Baby & Kids Fashion,breadcrumb_Eyewear,breadcrumb_Clutch,breadcrumb_Girl Bags,breadcrumb_Crossbody & Shoulder Bags,breadcrumb_Console Accessories,breadcrumb_Fashion Accessories,breadcrumb_Home Organizers,breadcrumb_Automobile Exterior Accessories,breadcrumb_Aksesoris,breadcrumb_Automotive,breadcrumb_Other Men Bags,breadcrumb_Casing & Protectors,breadcrumb_Tools,"breadcrumb_Folders, Paper Organizers & Accessories",breadcrumb_Non-Fiction Books,breadcrumb_Shoe Care & Accessories,breadcrumb_School & Office Equipment,breadcrumb_Bath & Body Care,breadcrumb_Dompet Koin,breadcrumb_Clutches,breadcrumb_Sports & Outdoor Accessories,breadcrumb_Prayer Attire & Equipment,"breadcrumb_Kabel, Charger, & Konverter",breadcrumb_Hobby & Collection,breadcrumb_Aksesoris Fashion,breadcrumb_Dompet Lipat,"breadcrumb_Cables, Chargers & Converters",breadcrumb_Art Supplies,breadcrumb_Handphone & Aksesoris Lainnya,breadcrumb_Backpacks,breadcrumb_Tas Anak Perempuan,breadcrumb_Other Women Bags,breadcrumb_Dompet Kartu,breadcrumb_Electronics,breadcrumb_Food Supplement,breadcrumb_Wallets,breadcrumb_Souvenir & Hadiah,breadcrumb_Camera Care,breadcrumb_Dompet Kunci & Handphone,breadcrumb_Tas Selempang & Bahu Wanita,breadcrumb_Men Muslim Wear,breadcrumb_Dompet Panjang,breadcrumb_Beauty Sets & Packages,breadcrumb_Men Bags,breadcrumb_Laptop Bags,breadcrumb_Other Women Shoes,breadcrumb_Tas Anak Laki-Laki,breadcrumb_Pouch Handphone,breadcrumb_Top-handle Bags,breadcrumb_Photography,breadcrumb_Makeup Accessories
0,20460325116,VONA Dompet Pria Lipat Pendek 2 Premium Kulit ...,4.79,126,145000.0,53000.0,19259,71,VONA Official Shop,"[""Shopee"",""Men Bags"",""Wallets"",""Bifold & Trifo...",4.76,111,100,1.0,6.0,113600,281,NaN,14893000.0,Solid,No Warranty,Synthetic Leather,No Warranty,Indonesia,Other,6,Tidak,Halus,Mengkilap,Kulit Sintesis,19259.0,KOTA JAKARTA SELATAN,4,0,10,1,0,92000.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
1,4116469787,AIRA WALLET JIMS HONEY Dompet Panjang Mewah Wa...,4.82,174,108000.0,75000.0,0,160,JIMS HONEY Deore.id,"[""Shopee"",""Women Bags"",""Wallets"",""Long Wallets""]",4.90,419,96,1.0,6.0,107500,304,NaN,22800000.0,None,No Warranty,Special Texture & Others,No Warranty,Tidak Diketahui,Other,0

In [199]:
df['Stok'].value_counts()

Stok
1.0         187
3.0         120
4.0         105
2.0         100
6.0          98
           ... 
212683.0      1
255.0         1
502.0         1
14775.0       1
874.0         1
Name: count, Length: 1351, dtype: int64

In [200]:
import pandas as pd
import re

def categorize_texture(val):
    if pd.isna(val):
        return 'Other'
    
    s = str(val).strip().lower()
    
    # Define keyword groups
    smooth = ['halus', 'lembut empuk']
    orange_peel = ['jeruk']
    crocodile = ['croco', 'buaya']
    embossed = ['timbul']
    woven = ['anyaman']
    layered = ['berlapis']
    synthetic = ['sintetis', 'sintesis']
    
    # Check each group in order (first match wins)
    if any(kw in s for kw in smooth):
        return 'Smooth'
    if any(kw in s for kw in orange_peel):
        return 'Orange Peel'
    if any(kw in s for kw in crocodile):
        return 'Crocodile'
    if any(kw in s for kw in embossed):
        return 'Embossed'
    if any(kw in s for kw in woven):
        return 'Woven'
    if any(kw in s for kw in layered):
        return 'Layered'
    if any(kw in s for kw in synthetic):
        return 'Synthetic'
    
    # Catch-all for remaining (e.g., '-', 'berkerikil', 'Motif Serat Kayu')
    return 'Other'

df['Tekstur Kulit'] = df['Tekstur Kulit'].apply(categorize_texture)

# Check distribution
print(df['Tekstur Kulit'].value_counts())

Tekstur Kulit
Other          5320
Smooth         2056
Embossed        410
Layered         240
Woven           172
Orange Peel      22
Crocodile        10
Synthetic         8
Name: count, dtype: int64


In [201]:
import pandas as pd
import re

def categorize_finish(val):
    if pd.isna(val):
        return 'Other'
    
    s = str(val).strip().lower()
    
    glossy = ['mengkilap']
    matte = ['matte', 'mate']
    
    has_glossy = any(kw in s for kw in glossy)
    has_matte = any(kw in s for kw in matte)
    
    if has_glossy and has_matte:
        return 'Mixed'
    elif has_glossy:
        return 'Glossy'
    elif has_matte:
        return 'Matte'
    else:
        return 'Other'
df['Tampilan Kulit'] = df['Tampilan Kulit'].apply(categorize_finish)

# Check distribution
print(df['Tampilan Kulit'].value_counts())

Tampilan Kulit
Other     5493
Matte     1856
Glossy     887
Mixed        2
Name: count, dtype: int64


In [202]:
import pandas as pd
import re

def categorize_leather_type(val):
    if pd.isna(val):
        return 'Uncategorized'
    
    s = str(val).strip().lower()
    
    # Synthetic keywords
    synthetic = ['sintetis', 'sintesis', 'pu', 'polyurethane', 'pvc', 'vegan', 'synthetic', 'puv']
    # Genuine keywords (animal + plain "kulit")
    genuine = ['sapi', 'domba', 'babi', 'bison', 'ular', 'hewan eksotis', 'kulit']
    # Non-leather / technical
    non_leather = ['kanvas', 'nilon', 'chocoly', 'bimo', 'polyester', 'anti air', 'non kulit']
    
    # Check synthetic first
    if any(kw in s for kw in synthetic):
        return 'Synthetic Leather'
    
    # Then genuine
    if any(kw in s for kw in genuine):
        return 'Genuine Leather'
    
    # Then non-leather
    if any(kw in s for kw in non_leather):
        return 'Non-Leather'
    
    # Everything else
    return 'Other'
df['Jenis Kulit']= df['Jenis Kulit'].apply(categorize_leather_type)

# Check distribution
print(df['Jenis Kulit'].value_counts())

Jenis Kulit
Uncategorized        6094
Other                1000
Synthetic Leather     696
Genuine Leather       384
Non-Leather            64
Name: count, dtype: int64


In [203]:
df['Stok'] = df['Stok'].fillna(0)
df['Stok'].value_counts()

Stok
0.0         1038
1.0          187
3.0          120
4.0          105
2.0          100
            ... 
212683.0       1
255.0          1
502.0          1
14775.0        1
874.0          1
Name: count, Length: 1351, dtype: int64

In [ ]:
import pandas as pd

# Example: create a dictionary mapping each unique value to a region.
# Since you have the exact list, you can build it manually or use a more dynamic approach.

region_map = {
    # Jabodetabek (or separate Jakarta)
    'KOTA JAKARTA SELATAN': 'Jabodetabek',
    'KOTA JAKARTA BARAT': 'Jabodetabek',
    'KOTA JAKARTA UTARA': 'Jabodetabek',
    'KOTA JAKARTA TIMUR': 'Jabodetabek',
    'KOTA JAKARTA PUSAT': 'Jabodetabek',
    'KOTA BEKASI': 'Jabodetabek',
    'KOTA DEPOK': 'Jabodetabek',
    'KOTA TANGERANG': 'Jabodetabek',
    'KOTA TANGERANG SELATAN': 'Jabodetabek',
    'KAB. BEKASI': 'Jabodetabek',
    'KAB. TANGERANG': 'Jabodetabek',
    'KAB. BOGOR': 'Jabodetabek',
    
    # Java (non-Jabodetabek) – add more as needed
    'KOTA BANDUNG': 'Java',
    'KAB. BANDUNG': 'Java',
    'KAB. BANDUNG BARAT': 'Java',
    'KOTA CIREBON': 'Java',
    'KOTA TASIKMALAYA': 'Java',
    'KAB. TASIKMALAYA': 'Java',
    'KOTA SUKABUMI': 'Java',
    'KAB. CIANJUR': 'Java',
    'KAB. GARUT': 'Java',
    'KAB. MAJALENGKA': 'Java',
    'KAB. PURBALINGGA': 'Java',
    'KAB. CILACAP': 'Java',
    'KAB. BANYUMAS': 'Java',
    'KAB. PEMALANG': 'Java',
    'KAB. PEKALONGAN': 'Java',
    'KOTA PEKALONGAN': 'Java',
    'KAB. BATANG': 'Java',
    'KAB. KENDAL': 'Java',
    'KOTA SEMARANG': 'Java',
    'KAB. DEMAK': 'Java',
    'KAB. REMBANG': 'Java',
    'KAB. BLORA': 'Java',
    'KAB. PATI': 'Java',
    'KAB. JEPARA': 'Java',
    'KAB. KUDUS': 'Java',
    'KOTA SURAKARTA (SOLO)': 'Java',
    'KAB. SUKOHARJO': 'Java',
    'KAB. KARANGANYAR': 'Java',
    'KAB. WONOGIRI': 'Java',
    'KOTA YOGYAKARTA': 'Java',
    'KAB. SLEMAN': 'Java',
    'KAB. BANTUL': 'Java',
    'KAB. GUNUNG KIDUL': 'Java',
    'KAB. KULON PROGO': 'Java',
    'KOTA MALANG': 'Java',
    'KAB. MALANG': 'Java',
    'KAB. PASURUAN': 'Java',
    'KOTA PASURUAN': 'Java',
    'KOTA PROBOLINGGO': 'Java',  # if appears
    'KAB. PROBOLINGGO': 'Java',
    'KOTA SURABAYA': 'Java',
    'KAB. SIDOARJO': 'Java',
    'KAB. GRESIK': 'Java',
    'KAB. LAMONGAN': 'Java',
    'KAB. BOJONEGORO': 'Java',
    'KAB. TUBAN': 'Java',
    'KAB. NGAWI': 'Java',
    'KAB. MADIUN': 'Java',
    'KOTA MADIUN': 'Java',
    'KAB. MAGETAN': 'Java',
    'KAB. PONOROGO': 'Java',
    'KAB. PACITAN': 'Java',
    'KAB. TRENGGALEK': 'Java',
    'KOTA BLITAR': 'Java',
    'KAB. BLITAR': 'Java',
    'KAB. KEDIRI': 'Java',
    'KOTA KEDIRI': 'Java',
    'KAB. NGANJUK': 'Java',
    'KAB. JOMBANG': 'Java',
    'KAB. MOJOKERTO': 'Java',
    'KOTA MOJOKERTO': 'Java',
    'KAB. BANGKALAN': 'Java',   # Madura (often grouped with East Java)
    'KAB. PAMEKASAN': 'Java',
    'KAB. SAMPANG': 'Java',
    'KAB. SUMENEP': 'Java',
    'KAB. SERANG': 'Java',       # Banten
    'KAB. PANDEGLANG': 'Java',
    'KAB. LEBAK': 'Java',
    'KOTA CILEGON': 'Java',
    
    # Sumatra
    'KOTA MEDAN': 'Sumatra',
    'KOTA PEKANBARU': 'Sumatra',
    'KOTA PALEMBANG': 'Sumatra',
    'KOTA PADANG': 'Sumatra',
    'KOTA BANDAR LAMPUNG': 'Sumatra',
    'KOTA PANGKAL PINANG': 'Sumatra',
    'KOTA TANJUNG PINANG': 'Sumatra',
    'KOTA BATAM': 'Sumatra',
    'KAB. KARIMUN': 'Sumatra',
    'KAB. BANGKA': 'Sumatra',
    # ... add more as needed
    
    # Kalimantan
    'KOTA BANJARMASIN': 'Kalimantan',
    'KOTA BANJARBARU': 'Kalimantan',
    'KAB. BANJAR': 'Kalimantan',
    'KAB. HULU SUNGAI TENGAH': 'Kalimantan',
    'KAB. HULU SUNGAI UTARA': 'Kalimantan',
    'KOTA BALIKPAPAN': 'Kalimantan',
    'KOTA SAMARINDA': 'Kalimantan',
    # ... add more
    
    # Sulawesi
    'KOTA MAKASSAR': 'Sulawesi',
    'KAB. GOWA': 'Sulawesi',
    'KAB. SOPPENG': 'Sulawesi',
    'KAB. SIDENRENG RAPPANG/RAPANG': 'Sulawesi',
    'KOTA MANADO': 'Sulawesi',
    # ... add more
    
    # Bali & Nusa Tenggara
    'KOTA DENPASAR': 'Bali & Nusa Tenggara',
    
    # Maluku & Papua
    # (none in your array, but could add)
    
    # Overseas
    'Luar Negeri': 'Overseas',
    
    # Unknown
    None: 'Unknown'
}

# Apply mapping
df['Dikirim Dari'] = df['Dikirim Dari'].map(region_map).fillna('Unknown')
df['Dikirim Dari'].value_counts()

Dikirim Dari
Jabodetabek             3938
Java                    2727
Unknown                 1215
Kalimantan               132
Sumatra                   89
Sulawesi                  87
Bali & Nusa Tenggara      42
Overseas                   8
Name: count, dtype: int64

In [205]:
df.brand.unique()

array([nan, 'Lock', 'JIMS HONEY', 'LOOK BACK', 'Baellerry', 'Menbense',
       'Sophie Paris', 'Ceka', 'Disney', 'CaseMe', 'LA LOUIS ANDREANO',
       'Walker Leather', 'Wallts Wallet Goods', 'Smile Goddess', 'CARDIN',
       'Fansy', 'Ratu Bilqis', 'smiggle', 'Samsung', 'Crocodile', 'PIOMA',
       'Oppo', 'Sony', 'CHIBAO', 'ANTAM', 'Miniso', 'Ray Clayn',
       'Forever Young'], dtype=object)

In [206]:
# Fill NaN first
df['brand'] = df['brand'].fillna('Unknown')

# Get value counts
brand_counts = df['brand'].value_counts()

# Define threshold, e.g., 0.1% of total rows
threshold = 0.001 * len(df)

# Create a new column: keep brand if count >= threshold, else 'Other'
df['brand'] = df['brand'].apply(lambda x: x if brand_counts[x] >= threshold else 'Other')

In [207]:
df.head()

,id,title,rating,reviews,initial_price,final_price,stock,favorite,seller_name,breadcrumb,seller_rating,seller_products,seller_chats_responded_percentage,seller_chat_time_reply,seller_joined_date,seller_followers,sold,brand,gmv_cal,Motif,Jenis Garansi,Bahan,Masa Garansi,Negara Asal,Penutup Dompet,Jumlah Slot Kartu,Slot Koin,Tekstur Kulit,Tampilan Kulit,Jenis Kulit,Stok,Dikirim Dari,variations_count,voucher_status,image_count,video_count,review,discount,breadcrumb_Mobile & Tablet Accessories,breadcrumb_Automotive Keychains & Key Covers,breadcrumb_Other Fashion Accessories,breadcrumb_Tote Bags,breadcrumb_Sports & Outdoor,breadcrumb_Other Sports & Outdoor,breadcrumb_Women Shoes,breadcrumb_Tas & Koper,breadcrumb_Aksesoris Make Up,breadcrumb_Umbrella,breadcrumb_Aksesoris Bayi & Anak,breadcrumb_Waist Bags & Chest Bags,breadcrumb_Motorcycle Rider Accessories,breadcrumb_Dompet Wanita,breadcrumb_Logam Mulia,breadcrumb_Sets,breadcrumb_Tas Wanita,breadcrumb_Women Bags,breadcrumb_Aksesoris Konsol,breadcrumb_Fashion Bayi & Anak,breadcrumb_Bag Accessories,breadcrumb_Handphone & Aksesoris,breadcrumb_Boy Bags,breadcrumb_Men Shoes,breadcrumb_Dompet,breadcrumb_Alat Kecantikan,breadcrumb_Ear Care,breadcrumb_Muslim Fashion,breadcrumb_Clutches & Wristlets,breadcrumb_Home & Living,breadcrumb_Health,breadcrumb_Aksesoris Fashion Lainnya,breadcrumb_Mobile & Accessories,breadcrumb_First Aid Supplies,breadcrumb_Eyes Makeup,breadcrumb_Other Photography,breadcrumb_Needlework,breadcrumb_Souvenir & Perlengkapan Pesta,breadcrumb_Stationery & Books,breadcrumb_Beauty & Care,breadcrumb_Men Clothes,breadcrumb_Casing & Skin,breadcrumb_Tas Pria,breadcrumb_Baby & Kids Fashion,breadcrumb_Eyewear,breadcrumb_Clutch,breadcrumb_Girl Bags,breadcrumb_Crossbody & Shoulder Bags,breadcrumb_Console Accessories,breadcrumb_Fashion Accessories,breadcrumb_Home Organizers,breadcrumb_Automobile Exterior Accessories,breadcrumb_Aksesoris,breadcrumb_Automotive,breadcrumb_Other Men Bags,breadcrumb_Casing & Protectors,breadcrumb_Tools,"breadcrumb_Folders, Paper Organizers & Accessories",breadcrumb_Non-Fiction Books,breadcrumb_Shoe Care & Accessories,breadcrumb_School & Office Equipment,breadcrumb_Bath & Body Care,breadcrumb_Dompet Koin,breadcrumb_Clutches,breadcrumb_Sports & Outdoor Accessories,breadcrumb_Prayer Attire & Equipment,"breadcrumb_Kabel, Charger, & Konverter",breadcrumb_Hobby & Collection,breadcrumb_Aksesoris Fashion,breadcrumb_Dompet Lipat,"breadcrumb_Cables, Chargers & Converters",breadcrumb_Art Supplies,breadcrumb_Handphone & Aksesoris Lainnya,breadcrumb_Backpacks,breadcrumb_Tas Anak Perempuan,breadcrumb_Other Women Bags,breadcrumb_Dompet Kartu,breadcrumb_Electronics,breadcrumb_Food Supplement,breadcrumb_Wallets,breadcrumb_Souvenir & Hadiah,breadcrumb_Camera Care,breadcrumb_Dompet Kunci & Handphone,breadcrumb_Tas Selempang & Bahu Wanita,breadcrumb_Men Muslim Wear,breadcrumb_Dompet Panjang,breadcrumb_Beauty Sets & Packages,breadcrumb_Men Bags,breadcrumb_Laptop Bags,breadcrumb_Other Women Shoes,breadcrumb_Tas Anak Laki-Laki,breadcrumb_Pouch Handphone,breadcrumb_Top-handle Bags,breadcrumb_Photography,breadcrumb_Makeup Accessories
0,20460325116,VONA Dompet Pria Lipat Pendek 2 Premium Kulit ...,4.79,126,145000.0,53000.0,19259,71,VONA Official Shop,"[""Shopee"",""Men Bags"",""Wallets"",""Bifold & Trifo...",4.76,111,100,1.0,6.0,113600,281,Unknown,14893000.0,Solid,No Warranty,Synthetic Leather,No Warranty,Indonesia,Other,6,Tidak,Smooth,Glossy,Synthetic Leather,19259.0,Jabodetabek,4,0,10,1,0,92000.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
1,4116469787,AIRA WALLET JIMS HONEY Dompet Panjang Mewah Wa...,4.82,174,108000.0,75000.0,0,160,JIMS HONEY Deore.id,"[""Shopee"",""Women Bags"",""Wallets"",""Long Wallets""]",4.90,419,96,1.0,6.0,107500,304,Unknown,22800000.0,None,No Warranty,Special Texture & Others,No Warranty,Tidak Diketahui,Other,0

# CONSOLIDATED DOMPET_CODE

In [ ]:
df=pd.read_csv("/Users/claranatalies/Documents/year 4/FYP/COMP4501-FYP/cleaned_dataset_final/Dompet_updated_cleaned.csv")
df.head()

In [209]:
import pandas as pd
import re
import numpy as np

# ------------------------------------------------------------------
# Constants
# ------------------------------------------------------------------
REF_DATE = pd.to_datetime('2026-03-21')               # Reference for seller join date
THRESHOLD_BRAND = 0.001                               # Relative frequency threshold for 'Other' brand

# ------------------------------------------------------------------
# Column‑specific cleaning functions
# ------------------------------------------------------------------

def clean_chat_time(val):
    """Convert seller_chat_time_reply to numeric hours."""
    if pd.isna(val):
        return None
    s = str(val).strip().lower()
    if s == 'hitungan jam':
        return 1.0
    if s == 'hitungan menit':
        return 0.5
    match = re.search(r'([\d,\.]+)\s*(jam|menit|hours?|hrs?)', s)
    if match:
        num_str = match.group(1).replace(',', '.')
        try:
            num = float(num_str)
        except:
            return None
        unit = match.group(2)
        if unit.startswith('jam') or unit.startswith('hour') or unit.startswith('hr'):
            return num
        elif unit.startswith('menit'):
            return num / 60.0
        else:
            return None
    try:
        return float(s)
    except:
        return None

def clean_seller_joined_date(val, ref_date=REF_DATE):
    """Convert seller_joined_date to years since reference date."""
    if pd.isna(val):
        return None

    def parse_absolute(s):
        s_clean = re.sub(r'\s*GMT.*$', '', s)
        s_clean = re.sub(r'\s*\(.*$', '', s_clean)
        try:
            return pd.to_datetime(s_clean, errors='coerce')
        except:
            return pd.NaT

    def relative_to_years(s):
        match = re.match(r'(\d+)\s+(tahun|bulan|hari)\s+lalu', s)
        if not match:
            return None
        num = int(match.group(1))
        unit = match.group(2)
        if unit == 'tahun':
            return float(num)
        elif unit == 'bulan':
            return num / 12.0
        else:  # hari
            return num / 365.25

    val_str = str(val).strip()
    if 'lalu' in val_str:
        years = relative_to_years(val_str)
        if years is not None:
            return years
    dt = parse_absolute(val_str)
    if pd.notna(dt):
        return (ref_date - dt).days / 365.25
    return None

def normalize_motif(s):
    """Helper: normalize motif string."""
    if pd.isna(s):
        return ''
    s = str(s).strip().lower()
    s = re.sub(r'[_-]', ' ', s)
    s = ' '.join(s.split())
    return s

# Motif keyword mapping (full dictionary as in your code)
keyword_to_category = {
    # Solid
    'polos': 'Solid',
    'solid color': 'Solid',
    'solid': 'Solid',
    'plain': 'Solid',
    'duo tone': 'Solid',
    'duotone': 'Solid',
    '2 warna': 'Solid',
    # Stripes
    'garis-garis': 'Stripes',
    'garis': 'Stripes',
    'stripe': 'Stripes',
    'stripes': 'Stripes',
    # Checks
    'kotak-kotak': 'Checks',
    'kotak kotak': 'Checks',
    'kotak': 'Checks',
    'plaid': 'Checks',
    'check': 'Checks',
    'houndstooth': 'Checks',
    # Floral
    'bunga-bunga': 'Floral',
    'bunga': 'Floral',
    'flower': 'Floral',
    'floral': 'Floral',
    'hawai': 'Floral',
    # Print/Logo
    'logo': 'Print/Logo',
    'logo jh': 'Print/Logo',
    'tulisan brand': 'Print/Logo',
    'grafik': 'Print/Logo',
    'graphic': 'Print/Logo',
    'print': 'Print/Logo',
    'sablon': 'Print/Logo',
    'bergambar': 'Print/Logo',
    'losantos': 'Print/Logo',
    'throwup': 'Print/Logo',
    'coach': 'Print/Logo',
    'independent': 'Print/Logo',
    'flag': 'Print/Logo',
    # Other (catch‑all)
    'loreng': 'Other',
    'croco': 'Other',
    'crocodile': 'Other',
    'kucing': 'Other',
    'bulu': 'Other',
    'embos': 'Other',
    'quilting': 'Other',
    'quiltet': 'Other',
    'kulit jeruk': 'Other',
    'jeruk': 'Other',
    'motif jeruk': 'Other',
    'urat kayu': 'Other',
    'serat kayu': 'Other',
    'carbon': 'Other',
    'tikar': 'Other',
    'full grain': 'Other',
    'kulit sintetis': 'Other',
    'tekstur': 'Other',
    'batik': 'Other',
    'songket': 'Other',
    'jacquard songket': 'Other',
    'primer jacquard': 'Other',
    'sulaman': 'Other',
    'hiasan': 'Other',
    'polkadot': 'Other',
    'sprinkle': 'Other',
    'mozaik': 'Other',
    'pattern': 'Other',
    'motive': 'Other',
    'doraemon': 'Other',
    'boneka': 'Other',
    'spiderman red': 'Other',
}

def get_motif_category(motif):
    """Return category for a single motif string."""
    if motif in keyword_to_category:
        return keyword_to_category[motif]
    for kw, cat in keyword_to_category.items():
        if kw in motif:
            return cat
    return 'Other'

def clean_motif_column(series):
    """Clean Motif column: take first motif, normalize, map to category."""
    norm = series.apply(normalize_motif)
    first_motif = norm.apply(lambda x: x.split(',')[0].strip() if x else '')
    return first_motif.apply(lambda x: get_motif_category(x) if x else None)

def clean_jenis_garansi(val):
    """Map warranty type to standard categories."""
    if pd.isna(val):
        return 'No Warranty'
    s = str(val).strip().lower()
    mapping = {
        'tidak garansi': 'No Warranty',
        'tidak ada': 'No Warranty',
        '-': 'No Warranty',
        '1 minggu': 'No Warranty',
        'garansi 100% original anker': 'No Warranty',
        'garansi supplier': 'Supplier Warranty',
        'garansi seller': 'Seller Warranty',
        'seller': 'Seller Warranty',
        'garansi toko': 'Seller Warranty',
        'garansi produsen': 'Manufacturer Warranty',
        'garansi internasional': 'International Warranty'
    }
    return mapping.get(s, 'No Warranty')

def categorize_material(val):
    """Categorise material into Genuine Leather, Synthetic Leather, Technical & Waterproof, etc."""
    if pd.isna(val):
        return 'Special Texture & Others'
    s = str(val).strip().lower()
    synthetic_indicators = ['sintetis', 'pu', 'synthetic', 'faux', 'polyurethane', 'polyutherane']
    if ('kulit' in s and not any(ind in s for ind in synthetic_indicators)) or \
       any(kw in s for kw in ['genuine', 'full grain', 'safiano', 'horse oil']):
        return 'Genuine Leather'
    synthetic_keywords = ['pu', 'sintetis', 'sintesis', 'synthetic', 'faux', 
                          'polyurethane', 'polyutherane', 'premium pu']
    if any(kw in s for kw in synthetic_keywords):
        return 'Synthetic Leather'
    tech_keywords = ['bimo', 'bino', 'chocoly', 'cordura', 'codura', 'duralite', 
                     'dinier', 'waterproof', 'ripstock']
    if any(kw in s for kw in tech_keywords):
        return 'Technical & Waterproof'
    fabric_keywords = ['kanvas', 'canvas', 'poliester', 'polyester', 'nilon', 'nylon', 
                       'katun', 'rayon', 'denim', 'sutra', 'brokat', 'tapis', 'felt', 'tekstil']
    if any(kw in s for kw in fabric_keywords):
        return 'Fabric & Textile'
    hardware_keywords = ['pvc', 'tpu', 'silikon', 'alumunium', 'alloy', 'metal', 
                         'plastik', 'mika', 'logam']
    if any(kw in s for kw in hardware_keywords):
        return 'Hardware & Hard Materials'
    other_keywords = ['croco', 'crocodile', 'kulit jeruk', 'bulu', 'urat kayu', 
                      'mozaik', 'bronze', 'balsam', 'lainnya', 'nan']
    if any(kw in s for kw in other_keywords):
        return 'Special Texture & Others'
    return 'Special Texture & Others'

def categorize_warranty(val):
    """Categorise warranty duration (if needed)."""
    if pd.isna(val):
        return 'No Warranty'
    s = str(val).strip().lower()
    no_warranty = [
        'tidak garansi', 'tidak ada', '0', 'tdk garansi',
        'tidak tersedia', 'tidak ada garansi'
    ]
    if any(kw in s for kw in no_warranty):
        return 'No Warranty'
    if 'shopee' in s or 'penjual' in s:
        return 'Short Term'
    match = re.search(r'(\d+(?:\.\d+)?)\s*(hari|jam|minggu|bulan|tahun)', s)
    if match:
        num = float(match.group(1))
        unit = match.group(2)
        if unit == 'jam':
            days = num / 24.0
        elif unit == 'hari':
            days = num
        elif unit == 'minggu':
            days = num * 7
        elif unit == 'bulan':
            days = num * 30.44
        else:  # tahun
            days = num * 365.25
        if unit in ['hari', 'jam'] and days <= 3:
            return 'Immediate'
        elif unit == 'minggu':
            return 'Short Term'
        elif unit == 'hari' and days > 3:
            return 'Short Term'
        elif unit == 'bulan' and num <= 6:
            return 'Medium Term'
        elif unit == 'bulan' and num > 6:
            return 'Long Term'
        elif unit == 'tahun':
            return 'Long Term'
    return 'No Warranty'

def categorize_closure(val):
    """Categorise closure type (Zipper, Button, Magnetic, etc.)."""
    if pd.isna(val):
        return 'Other'
    s = str(val).strip().lower()
    zipper = ['resleting', 'risleting', 'zip']
    button = ['kancing']
    magnetic = ['magnet', 'magnetic']
    snap_hook = ['kunci', 'pin', 'klip', 'gawang', 'tutup klip']
    velcro = ['velcro']
    other = ['lipat', 'lainnya']
    matched = []
    if any(kw in s for kw in zipper):
        matched.append('Zipper')
    if any(kw in s for kw in button):
        matched.append('Button')
    if any(kw in s for kw in magnetic):
        matched.append('Magnetic')
    if any(kw in s for kw in snap_hook):
        matched.append('Snap/Hook/Lock')
    if any(kw in s for kw in velcro):
        matched.append('Velcro')
    if len(matched) > 1:
        return 'Combination'
    elif len(matched) == 1:
        return matched[0]
    else:
        return 'Other' if any(kw in s for kw in other) else 'Other'

def categorize_texture(val):
    """Categorise leather texture."""
    if pd.isna(val):
        return 'Other'
    s = str(val).strip().lower()
    smooth = ['halus', 'lembut empuk']
    orange_peel = ['jeruk']
    crocodile = ['croco', 'buaya']
    embossed = ['timbul']
    woven = ['anyaman']
    layered = ['berlapis']
    synthetic = ['sintetis', 'sintesis']
    if any(kw in s for kw in smooth):
        return 'Smooth'
    if any(kw in s for kw in orange_peel):
        return 'Orange Peel'
    if any(kw in s for kw in crocodile):
        return 'Crocodile'
    if any(kw in s for kw in embossed):
        return 'Embossed'
    if any(kw in s for kw in woven):
        return 'Woven'
    if any(kw in s for kw in layered):
        return 'Layered'
    if any(kw in s for kw in synthetic):
        return 'Synthetic'
    return 'Other'

def categorize_finish(val):
    """Categorise leather finish (Glossy/Matte/Mixed)."""
    if pd.isna(val):
        return 'Other'
    s = str(val).strip().lower()
    glossy = ['mengkilap']
    matte = ['matte', 'mate']
    has_glossy = any(kw in s for kw in glossy)
    has_matte = any(kw in s for kw in matte)
    if has_glossy and has_matte:
        return 'Mixed'
    elif has_glossy:
        return 'Glossy'
    elif has_matte:
        return 'Matte'
    else:
        return 'Other'

def categorize_leather_type(val):
    """Categorise leather type (Synthetic, Genuine, Non‑Leather, Other)."""
    if pd.isna(val):
        return 'Uncategorized'
    s = str(val).strip().lower()
    synthetic = ['sintetis', 'sintesis', 'pu', 'polyurethane', 'pvc', 'vegan', 'synthetic', 'puv']
    genuine = ['sapi', 'domba', 'babi', 'bison', 'ular', 'hewan eksotis', 'kulit']
    non_leather = ['kanvas', 'nilon', 'chocoly', 'bimo', 'polyester', 'anti air', 'non kulit']
    if any(kw in s for kw in synthetic):
        return 'Synthetic Leather'
    if any(kw in s for kw in genuine):
        return 'Genuine Leather'
    if any(kw in s for kw in non_leather):
        return 'Non-Leather'
    return 'Other'

# ------------------------------------------------------------------
# Location mapping (partial – you must complete this dictionary)
# ------------------------------------------------------------------
region_map = {
    # Jabodetabek
    'KOTA JAKARTA SELATAN': 'Jabodetabek',
    'KOTA JAKARTA BARAT': 'Jabodetabek',
    'KOTA JAKARTA UTARA': 'Jabodetabek',
    'KOTA JAKARTA TIMUR': 'Jabodetabek',
    'KOTA JAKARTA PUSAT': 'Jabodetabek',
    'KOTA BEKASI': 'Jabodetabek',
    'KOTA DEPOK': 'Jabodetabek',
    'KOTA TANGERANG': 'Jabodetabek',
    'KOTA TANGERANG SELATAN': 'Jabodetabek',
    'KAB. BEKASI': 'Jabodetabek',
    'KAB. TANGERANG': 'Jabodetabek',
    'KAB. BOGOR': 'Jabodetabek',
    # Java (non‑Jabodetabek)
    'KOTA BANDUNG': 'Java',
    'KAB. BANDUNG': 'Java',
    'KAB. BANDUNG BARAT': 'Java',
    'KOTA CIREBON': 'Java',
    'KOTA TASIKMALAYA': 'Java',
    'KAB. TASIKMALAYA': 'Java',
    'KOTA SUKABUMI': 'Java',
    'KAB. CIANJUR': 'Java',
    'KAB. GARUT': 'Java',
    'KAB. MAJALENGKA': 'Java',
    'KAB. PURBALINGGA': 'Java',
    'KAB. CILACAP': 'Java',
    'KAB. BANYUMAS': 'Java',
    'KAB. PEMALANG': 'Java',
    'KAB. PEKALONGAN': 'Java',
    'KOTA PEKALONGAN': 'Java',
    'KAB. BATANG': 'Java',
    'KAB. KENDAL': 'Java',
    'KOTA SEMARANG': 'Java',
    'KAB. DEMAK': 'Java',
    'KAB. REMBANG': 'Java',
    'KAB. BLORA': 'Java',
    'KAB. PATI': 'Java',
    'KAB. JEPARA': 'Java',
    'KAB. KUDUS': 'Java',
    'KOTA SURAKARTA (SOLO)': 'Java',
    'KAB. SUKOHARJO': 'Java',
    'KAB. KARANGANYAR': 'Java',
    'KAB. WONOGIRI': 'Java',
    'KOTA YOGYAKARTA': 'Java',
    'KAB. SLEMAN': 'Java',
    'KAB. BANTUL': 'Java',
    'KAB. GUNUNG KIDUL': 'Java',
    'KAB. KULON PROGO': 'Java',
    'KOTA MALANG': 'Java',
    'KAB. MALANG': 'Java',
    'KAB. PASURUAN': 'Java',
    'KOTA PASURUAN': 'Java',
    'KOTA PROBOLINGGO': 'Java',
    'KAB. PROBOLINGGO': 'Java',
    'KOTA SURABAYA': 'Java',
    'KAB. SIDOARJO': 'Java',
    'KAB. GRESIK': 'Java',
    'KAB. LAMONGAN': 'Java',
    'KAB. BOJONEGORO': 'Java',
    'KAB. TUBAN': 'Java',
    'KAB. NGAWI': 'Java',
    'KAB. MADIUN': 'Java',
    'KOTA MADIUN': 'Java',
    'KAB. MAGETAN': 'Java',
    'KAB. PONOROGO': 'Java',
    'KAB. PACITAN': 'Java',
    'KAB. TRENGGALEK': 'Java',
    'KOTA BLITAR': 'Java',
    'KAB. BLITAR': 'Java',
    'KAB. KEDIRI': 'Java',
    'KOTA KEDIRI': 'Java',
    'KAB. NGANJUK': 'Java',
    'KAB. JOMBANG': 'Java',
    'KAB. MOJOKERTO': 'Java',
    'KOTA MOJOKERTO': 'Java',
    'KAB. BANGKALAN': 'Java',
    'KAB. PAMEKASAN': 'Java',
    'KAB. SAMPANG': 'Java',
    'KAB. SUMENEP': 'Java',
    'KAB. SERANG': 'Java',
    'KAB. PANDEGLANG': 'Java',
    'KAB. LEBAK': 'Java',
    'KOTA CILEGON': 'Java',
    # Sumatra
    'KOTA MEDAN': 'Sumatra',
    'KOTA PEKANBARU': 'Sumatra',
    'KOTA PALEMBANG': 'Sumatra',
    'KOTA PADANG': 'Sumatra',
    'KOTA BANDAR LAMPUNG': 'Sumatra',
    'KOTA PANGKAL PINANG': 'Sumatra',
    'KOTA TANJUNG PINANG': 'Sumatra',
    'KOTA BATAM': 'Sumatra',
    'KAB. KARIMUN': 'Sumatra',
    'KAB. BANGKA': 'Sumatra',
    # Kalimantan
    'KOTA BANJARMASIN': 'Kalimantan',
    'KOTA BANJARBARU': 'Kalimantan',
    'KAB. BANJAR': 'Kalimantan',
    'KAB. HULU SUNGAI TENGAH': 'Kalimantan',
    'KAB. HULU SUNGAI UTARA': 'Kalimantan',
    'KOTA BALIKPAPAN': 'Kalimantan',
    'KOTA SAMARINDA': 'Kalimantan',
    # Sulawesi
    'KOTA MAKASSAR': 'Sulawesi',
    'KAB. GOWA': 'Sulawesi',
    'KAB. SOPPENG': 'Sulawesi',
    'KAB. SIDENRENG RAPPANG/RAPANG': 'Sulawesi',
    'KOTA MANADO': 'Sulawesi',
    # Bali & Nusa Tenggara
    'KOTA DENPASAR': 'Bali & Nusa Tenggara',
    # Overseas
    'Luar Negeri': 'Overseas',
    # Unknown
    None: 'Unknown'
}

# ------------------------------------------------------------------
# Main cleaning pipeline
# ------------------------------------------------------------------
def clean_dataframe(df):
    """Apply all cleaning transformations to the input DataFrame."""
    df = df.copy()

    # 1. Seller chat time and join date
    df['seller_chat_time_reply'] = df['seller_chat_time_reply'].apply(clean_chat_time)
    df['seller_joined_date'] = df['seller_joined_date'].apply(clean_seller_joined_date)

    # 2. Motif (pattern)
    df['Motif'] = clean_motif_column(df['Motif'])

    # 3. Warranty type
    df['Jenis Garansi'] = df['Jenis Garansi'].apply(clean_jenis_garansi)

    # 4. Warranty duration (Masa Garansi)
    # Note: The column name used in your original code was 'Masa Garansi'
    if 'Masa Garansi' in df.columns:
        df['Masa Garansi'] = df['Masa Garansi'].apply(categorize_warranty)

    # 5. Material (Bahan)
    df['Bahan'] = df['Bahan'].apply(categorize_material)

    # 6. Closure (Penutup Dompet)
    df['Penutup Dompet'] = df['Penutup Dompet'].apply(categorize_closure)

    # 7. Leather texture (Tekstur Kulit)
    df['Tekstur Kulit'] = df['Tekstur Kulit'].apply(categorize_texture)

    # 8. Leather finish (Tampilan Kulit)
    df['Tampilan Kulit'] = df['Tampilan Kulit'].apply(categorize_finish)

    # 9. Leather type (Jenis Kulit)
    df['Jenis Kulit'] = df['Jenis Kulit'].apply(categorize_leather_type)

    # 10. Simple fill operations
    df['Jumlah Slot Kartu'] = df['Jumlah Slot Kartu'].fillna(0)
    df['Slot Koin'] = df['Slot Koin'].fillna('Tidak')
    df['Stok'] = df['Stok'].fillna(0)
    df['Negara Asal'] = df['Negara Asal'].fillna('Unknown')

    # 11. Brand grouping by frequency
    df['brand'] = df['brand'].fillna('Unknown')
    brand_counts = df['brand'].value_counts()
    thresh = THRESHOLD_BRAND * len(df)
    df['brand'] = df['brand'].apply(lambda x: x if brand_counts[x] >= thresh else 'Other')

    # 12. Location (Dikirim Dari) – using region mapping
    df['Dikirim Dari'] = df['Dikirim Dari'].map(region_map).fillna('Unknown')

    return df

# ------------------------------------------------------------------
# Execution
# ------------------------------------------------------------------
if __name__ == "__main__":
    # Replace with your actual file path
    input_file = '/Users/claranatalies/Documents/year 4/FYP/COMP4501-FYP/cleaned_dataset_final/Dompet_updated_cleaned.csv'
    output_file = 'cleaned_data.csv'

    df_raw = pd.read_csv(input_file)
    df_cleaned = clean_dataframe(df_raw)
    df_cleaned.to_csv(output_file, index=False)

    print("Cleaning complete. Output saved to:", output_file)
    print("Shape after cleaning:", df_cleaned.shape)
    print(df_cleaned.head())

/var/folders/8z/b64380y12kv332c9qsmc494w0000gq/T/ipykernel_62833/2422224902.py:545: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(input_file)


Cleaning complete. Output saved to: cleaned_data.csv
Shape after cleaning: (8238, 133)
            id                                              title  rating  \
0  20460325116  VONA Dompet Pria Lipat Pendek 2 Premium Kulit ...    4.79   
1   4116469787  AIRA WALLET JIMS HONEY Dompet Panjang Mewah Wa...    4.82   
2  13197858916  [Lms] Dompet Pria Impor Kulit PU Premium Resle...    4.88   
3  11706914967  Baellerry RFID Anti Theft Card Dompet Kartu Wa...    4.83   
4   3406406326  Wallts Aster - Dompet Wallet Kartu Pria dan Wa...    4.93   

   reviews  initial_price  final_price  stock  favorite  \
0      126       145000.0      53000.0  19259        71   
1      174       108000.0      75000.0      0       160   
2      169        89000.0      39800.0    107       150   
3     1886        51000.0      42500.0    188        23   
4     2872       259900.0     125000.0  10093        47   

                         seller_name  \
0                 VONA Official Shop   
1              

In [211]:
# After cleaning, display value counts for all columns
for col in df_cleaned.columns:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print('-'*50)
    
    # Get value counts (including NaN)
    vc = df_cleaned[col].value_counts(dropna=False)
    
    # If there are more than 30 unique values, show only top 30
    if len(vc) > 30:
        print(f"Top 30 out of {len(vc)} unique values:\n")
        print(vc.head(30))
    else:
        print(vc)
    
    print(f"\nTotal rows: {len(df_cleaned)} | Unique values: {len(vc)}")


Column: id
--------------------------------------------------
Top 30 out of 4190 unique values:

id
2832363304     3
13401888664    3
13845770138    3
20460325116    2
17692355387    2
12775230301    2
4837744112     2
13695238184    2
23008288333    2
21781156862    2
6090628535     2
23521999728    2
3037437068     2
22174880237    2
18390188711    2
13921641714    2
1868915182     2
20133809276    2
11898737598    2
1848482388     2
9424580769     2
4008474648     2
11432452536    2
23969538050    2
6932995609     2
16792306816    2
19842743007    2
2039650018     2
18845914825    2
6942209265     2
Name: count, dtype: int64

Total rows: 8238 | Unique values: 4190

Column: title
--------------------------------------------------
Top 30 out of 4130 unique values:

title
JIMS HONEY DOMPET LIPAT WANITA CARMEL WALLET                                                                                                                                                                            

In [221]:
import ast
import pandas as pd

def parse_breadcrumb(val):
    """Convert string representation of a list to a Python list."""
    if pd.isna(val):
        return []
    try:
        # Use ast.literal_eval to safely parse the string
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        # If parsing fails, return an empty list
        return []

def categorize_breadcrumb(val):
    """Categorise product type from breadcrumb path."""
    path = parse_breadcrumb(val)
    if not path:
        return 'Unknown'

    # Remove leading "Shopee" if present (often the first element)
    if path and path[0] == 'Shopee':
        path = path[1:]

    # Convert to lower case for easier matching
    path_lower = [p.lower() for p in path]

    # Helper to detect gender
    def is_male(path_str):
        male_keywords = ['pria', 'men', 'laki', 'cowok', 'boy']
        return any(kw in path_str for kw in male_keywords)

    def is_female(path_str):
        female_keywords = ['wanita', 'women', 'cewek', 'girl']
        return any(kw in path_str for kw in female_keywords)

    # Determine if it's a wallet
    is_wallet = any('dompet' in p for p in path_lower) or any('wallet' in p for p in path_lower)

    # Determine if it's a bag (but not wallet)
    is_bag = any('tas' in p for p in path_lower) or any('bag' in p for p in path_lower)

    # Determine if it's a phone case / accessory
    is_phone = any('phone' in p for p in path_lower) or any('handphone' in p for p in path_lower) or \
               any('casing' in p for p in path_lower) or any('skin' in p for p in path_lower)

    # Determine if it's an accessory (other than phone)
    is_accessory = any('aksesoris' in p for p in path_lower) or any('accessories' in p for p in path_lower)

    # Now decide category
    if is_wallet:
        # Try to infer gender
        for p in path_lower:
            if is_male(p):
                return 'Men\'s Wallets'
            if is_female(p):
                return 'Women\'s Wallets'
        # Fallback if gender not clear
        return 'Wallets'

    elif is_bag:
        for p in path_lower:
            if is_male(p):
                return 'Men\'s Bags'
            if is_female(p):
                return 'Women\'s Bags'
        return 'Bags'

    elif is_phone:
        return 'Phone Cases & Accessories'

    elif is_accessory:
        return 'Accessories'

    else:
        return 'Other'

In [214]:
df['breadcrumb_group'] = df['breadcrumb'].apply(categorize_breadcrumb)

# Check distribution
print(df['breadcrumb_group'].value_counts())

breadcrumb_group
Men's Wallets                6983
Men's Bags                    415
Phone Cases & Accessories     401
Other                         248
Accessories                    96
Women's Wallets                75
Wallets                         8
Women's Bags                    7
Bags                            5
Name: count, dtype: int64


In [223]:
non_wallet_breadcrumbs = df[~df['breadcrumb'].str.contains('Dompet|Wallets')]
non_wallet_breadcrumbs.head()

,id,title,rating,reviews,initial_price,final_price,stock,favorite,seller_name,breadcrumb,seller_rating,seller_products,seller_chats_responded_percentage,seller_chat_time_reply,seller_joined_date,seller_followers,sold,brand,gmv_cal,Motif,Jenis Garansi,Bahan,Masa Garansi,Negara Asal,Penutup Dompet,Jumlah Slot Kartu,Slot Koin,Tekstur Kulit,Tampilan Kulit,Jenis Kulit,Stok,Dikirim Dari,variations_count,voucher_status,image_count,video_count,review,discount,breadcrumb_Mobile & Tablet Accessories,breadcrumb_Automotive Keychains & Key Covers,breadcrumb_Other Fashion Accessories,breadcrumb_Tote Bags,breadcrumb_Sports & Outdoor,breadcrumb_Other Sports & Outdoor,breadcrumb_Women Shoes,breadcrumb_Tas & Koper,breadcrumb_Aksesoris Make Up,breadcrumb_Umbrella,breadcrumb_Aksesoris Bayi & Anak,breadcrumb_Waist Bags & Chest Bags,breadcrumb_Motorcycle Rider Accessories,breadcrumb_Dompet Wanita,breadcrumb_Logam Mulia,breadcrumb_Sets,breadcrumb_Tas Wanita,breadcrumb_Women Bags,breadcrumb_Aksesoris Konsol,breadcrumb_Fashion Bayi & Anak,breadcrumb_Bag Accessories,breadcrumb_Handphone & Aksesoris,breadcrumb_Boy Bags,breadcrumb_Men Shoes,breadcrumb_Dompet,breadcrumb_Alat Kecantikan,breadcrumb_Ear Care,breadcrumb_Muslim Fashion,breadcrumb_Clutches & Wristlets,breadcrumb_Home & Living,breadcrumb_Health,breadcrumb_Aksesoris Fashion Lainnya,breadcrumb_Mobile & Accessories,breadcrumb_First Aid Supplies,breadcrumb_Eyes Makeup,breadcrumb_Other Photography,breadcrumb_Needlework,breadcrumb_Souvenir & Perlengkapan Pesta,breadcrumb_Stationery & Books,breadcrumb_Beauty & Care,breadcrumb_Men Clothes,breadcrumb_Casing & Skin,breadcrumb_Tas Pria,breadcrumb_Baby & Kids Fashion,breadcrumb_Eyewear,breadcrumb_Clutch,breadcrumb_Girl Bags,breadcrumb_Crossbody & Shoulder Bags,breadcrumb_Console Accessories,breadcrumb_Fashion Accessories,breadcrumb_Home Organizers,breadcrumb_Automobile Exterior Accessories,breadcrumb_Aksesoris,breadcrumb_Automotive,breadcrumb_Other Men Bags,breadcrumb_Casing & Protectors,breadcrumb_Tools,"breadcrumb_Folders, Paper Organizers & Accessories",breadcrumb_Non-Fiction Books,breadcrumb_Shoe Care & Accessories,breadcrumb_School & Office Equipment,breadcrumb_Bath & Body Care,breadcrumb_Dompet Koin,breadcrumb_Clutches,breadcrumb_Sports & Outdoor Accessories,breadcrumb_Prayer Attire & Equipment,"breadcrumb_Kabel, Charger, & Konverter",breadcrumb_Hobby & Collection,breadcrumb_Aksesoris Fashion,breadcrumb_Dompet Lipat,"breadcrumb_Cables, Chargers & Converters",breadcrumb_Art Supplies,breadcrumb_Handphone & Aksesoris Lainnya,breadcrumb_Backpacks,breadcrumb_Tas Anak Perempuan,breadcrumb_Other Women Bags,breadcrumb_Dompet Kartu,breadcrumb_Electronics,breadcrumb_Food Supplement,breadcrumb_Wallets,breadcrumb_Souvenir & Hadiah,breadcrumb_Camera Care,breadcrumb_Dompet Kunci & Handphone,breadcrumb_Tas Selempang & Bahu Wanita,breadcrumb_Men Muslim Wear,breadcrumb_Dompet Panjang,breadcrumb_Beauty Sets & Packages,breadcrumb_Men Bags,breadcrumb_Laptop Bags,breadcrumb_Other Women Shoes,breadcrumb_Tas Anak Laki-Laki,breadcrumb_Pouch Handphone,breadcrumb_Top-handle Bags,breadcrumb_Photography,breadcrumb_Makeup Accessories,breadcrumb_group
5,14600249323,Tas Selempang Pria dan Wanita Slingbag Casual ...,4.81,67,85000.0,23000.0,7,25,willgoodme official shop,"[""Shopee"",""Men Bags"",""Crossbody & Shoulder Bags""]",4.63,62,100,1.0,5.0,2900,138,Unknown,3174000.0,None,No Warranty,Special Texture & Others,No Warranty,Tidak Diketahui,Other,0,Tidak,Other,Other,Uncategorized,7.0,Java,6,0,8,0,0,62000.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,Men's Bags
11,7838008415,Samsung galaxy S7 / S7 Flat Flip Case Caseme C...,4.89,18,90000.0,83700.0,997,14,VinVend ACC,"[""Shopee"",""Mobile & Accessories"",""Casing & Pro...",4.83,511,93,1.0,6.0,5700,28,Unknown,2343600.0,None,No Warranty,Synthetic Leather,No Warranty,Tidak Diketahui,Other,0,Tid